<a href="https://colab.research.google.com/github/huanbv/BVU.CaoHoc.BuiVanHuan-TriTueNhanTaoNangCao_UngDung/blob/main/Demo_Gradio_CPU_Nhom1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo Gradio — Nhóm 1 (chạy riêng, dùng được CPU)
**Dùng khi hết GPU miễn phí.** Runtime > Change runtime type > **CPU** > Save, rồi **Runtime > Run all** (Ctrl+F9).

- Chỉ cần thư mục `vehicle-detection` trên Google Drive (có `weights/yolov8_best.pt`, `unet_aug_best.pth`, `resnet50_vehicle_v2.pth`).
- Lần đầu mất khoảng 1–2 phút để cài thư viện và nạp mô hình. Trên CPU mỗi ảnh mất vài giây.
- Chạy xong, bấm link `https://....gradio.live` ở cuối Cell 4 hoặc dùng giao diện ngay trong notebook.


In [1]:
# CELL 1: Cài thư viện + kết nối Drive (khoảng 1 phút)
!pip install -q ultralytics gradio "fast-plate-ocr[onnx]"
import os, time, json, glob, warnings
from pathlib import Path
import numpy as np, cv2, torch, torch.nn as nn
from torchvision import transforms, models
from PIL import Image
from ultralytics import YOLO
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(max(1, os.cpu_count() or 1))
print('Device:', DEVICE)

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = None
for c in ['/content/drive/MyDrive/vehicle-detection', '/content/drive/MyDrive/Project_Traffic_AI/vehicle-detection']:
    if Path(c).exists():
        BASE_DIR = Path(c); break
if BASE_DIR is None:
    hits = glob.glob('/content/drive/MyDrive/**/vehicle-detection/weights', recursive=True)
    BASE_DIR = Path(hits[0]).parent if hits else None
assert BASE_DIR is not None, 'Khong thay thu muc vehicle-detection tren Drive'
WEIGHTS_DIR = BASE_DIR / 'weights'
TEST_IMG_DIR = BASE_DIR / 'dataset' / 'test' / 'images'
CLASS_NAMES = ['Car', 'Motorbike', 'Truck']
NUM_CLASSES = 3
for f in ['yolov8_best.pt', 'unet_aug_best.pth', 'resnet50_vehicle_v2.pth']:
    print(' ', f, 'OK' if (WEIGHTS_DIR / f).exists() else 'THIEU!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 29.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Device: cpu
Mounted at /content/drive
  yolov8_best.pt OK
  unet_aug_best.pth OK
  resnet50_vehicle_v2.pth OK


In [2]:
# CELL 2: Kiến trúc U-Net và ResNet50 (giống notebook chính)
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class UNetSegmentation(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = ConvBlock(3, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ConvBlock(128, 256)

        # Decoder
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(256, 128)  # 128 (up) + 128 (skip) = 256 in
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)   # 64 (up) + 64 (skip) = 128 in

        # Output
        self.out_conv = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        # Bottleneck
        b = self.bottleneck(self.pool2(e2))
        # Decoder
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return torch.sigmoid(self.out_conv(d1))


class VehicleResNet50(nn.Module):
    """ResNet50 TN1 v2. weights=None: không tải ImageNet vì file _v2.pth đã chứa toàn bộ mô hình."""
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.fc = nn.Sequential(nn.Linear(2048, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.backbone(x)

print('Đã định nghĩa U-Net và ResNet50.')

Đã định nghĩa U-Net và ResNet50.


In [3]:
# CELL 3: OCR biển số v3.1 (fast-plate-ocr + làm phẳng + chỉnh sáng/tối)
# ===== Cell 5.4c: Nạp OCR biển số v3.1 — fast-plate-ocr + làm phẳng theo ký tự + chỉnh sáng/tối =====
# Đặt TRƯỚC ô 5.5. Các ô 5.5, 5.5b, 5.5b-2, 5.5b-3, 5.5b-4, 5.5c, 5.6 (Gradio) đều gọi recognize_plate_text() bản này.
# Thử trên 12 biển có đáp án (máy cá nhân): EasyOCR 0-1/12 đúng, fast-plate-ocr 12/12 đúng, ~6 ms/biển (CPU).
# Thử độ bền 13 biển x 8 kịch bản x 5 lần (nghiêng, xoay, phối cảnh, tối, cháy sáng):
#   đọc thẳng 66.3% -> v3.1 73.8%; phối cảnh mạnh 0% -> 38.5%; xoay 30° 53.8% -> 47.7% (kém hơn chút).
import re, difflib
from fast_plate_ocr import LicensePlateRecognizer

PLATE_OCR = LicensePlateRecognizer('cct-xs-v2-global-model', device='cpu')
_PLATE_GRAY = 'gray' in str(getattr(PLATE_OCR.config, 'image_color_mode', 'rgb')).lower()
_CAR = re.compile(r'^(\d{2}[A-Z]{1,2})(\d{4,5})$')      # 1 dòng: 51F-324.88, 30LD-1234
_MOTO = re.compile(r'^(\d{2}[A-Z][A-Z0-9])(\d{4,5})$')  # 2 dòng: 59C3-123.45

def _fmt_plate(s):
    """Chèn dấu theo định dạng biển dân sự VN nếu khớp; biển quân sự/ngoại giao... giữ nguyên."""
    for pat in (_CAR, _MOTO):
        m = pat.match(s)
        if m:
            d = m.group(2)
            return f"{m.group(1)}-{d[:3]}.{d[3:]}" if len(d) == 5 else f"{m.group(1)}-{d}"
    return s


# ---------- Làm phẳng biển số theo vị trí ký tự + RANSAC (thay cho tìm viền kiểu CamScanner) ----------
def _char_boxes(gray):
    """Tìm các khối có hình dạng giống ký tự (thử cả chữ tối/nền sáng và chữ sáng/nền tối)."""
    H, W = gray.shape
    out = []
    for inv in (False, True):
        g = 255 - gray if inv else gray
        th = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
                                   max(11, (min(H, W) // 4) | 1), 8)
        _, _, st, _ = cv2.connectedComponentsWithStats(th, 8)
        boxes = [(x, y, w, h) for x, y, w, h, a in st[1:]
                 if max(8, 0.06 * H) < h < 0.9 * H and 0.08 < w / h < 1.0 and a > 0.25 * w * h
                 and x > 1 and y > 1 and x + w < W - 1 and y + h < H - 1]
        out.append(np.array(boxes, dtype=np.float32).reshape(-1, 4))
    return out


def _best_row(b):
    """RANSAC: nhóm khối lớn nhất nằm trên 1 đường thẳng, chiều cao tương đồng, xếp liền nhau thành chuỗi."""
    if len(b) < 3:
        return np.zeros(len(b), bool)
    cx, cy, h = b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2, b[:, 3]
    best = np.zeros(len(b), bool)
    for i in range(len(b)):
        for j in range(i + 1, len(b)):
            if abs(cx[j] - cx[i]) < 1e-3:
                continue
            k = (cy[j] - cy[i]) / (cx[j] - cx[i])
            if abs(k) > 1.0:                                                # dòng chữ nghiêng tối đa ~45°
                continue
            hm = (h[i] + h[j]) / 2
            inl = (np.abs(cy - (cy[i] + k * (cx - cx[i]))) < 0.3 * hm) & (h > 0.4 * hm) & (h < 2.5 * hm)
            idx = np.where(inl)[0]
            if len(idx) < 3:
                continue
            idx = idx[np.argsort(cx[idx])]                                  # chỉ giữ chuỗi liền nhau dài nhất
            gaps = np.diff(cx[idx]) > 1.8 * np.maximum(h[idx][:-1], h[idx][1:])
            chain = max(np.split(idx, np.where(gaps)[0] + 1), key=len)
            if len(chain) > best.sum():
                best = np.zeros(len(b), bool); best[chain] = True
    return best


def _fit(xs, ys):
    if len(xs) >= 2 and np.ptp(xs) > 0:
        return np.polyfit(xs, ys, 1)
    return np.array([0.0, float(np.mean(ys))])


def rectify_by_chars(img_rgb, out_h=96):
    """Trả ảnh biển đã làm phẳng, hoặc None nếu không tìm được dòng chữ đủ tin cậy."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    scale = 1.0
    if gray.shape[0] < 120:                                             # phóng to để tách ký tự rõ hơn
        scale = 120.0 / gray.shape[0]
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.createCLAHE(2.0, (8, 8)).apply(gray)
    rows, best_n = None, 0
    for b in _char_boxes(gray):                                         # thử 2 cực tính, giữ bên có nhiều ký tự hơn
        m1 = _best_row(b)
        if m1.sum() < 4:
            continue
        r1, rest = b[m1], b[~m1]
        m2 = _best_row(rest) if len(rest) >= 3 else np.zeros(len(rest), bool)
        r2 = None
        if m2.sum() >= 2:                                               # dòng thứ 2 (biển vuông / xe máy)
            c = rest[m2]
            h1 = np.median(r1[:, 3])
            dy = np.median(c[:, 1] + c[:, 3] / 2) - np.median(r1[:, 1] + r1[:, 3] / 2)
            ov = min((c[:, 0] + c[:, 2]).max(), (r1[:, 0] + r1[:, 2]).max()) - max(c[:, 0].min(), r1[:, 0].min())
            if 0.8 * h1 < abs(dy) < 2.2 * h1 and ov > 0:
                r2 = c
        n = len(r1) + (0 if r2 is None else len(r2))
        if n > best_n:
            rows, best_n = (r1, r2), n
    if rows is None:
        return None
    r1, r2 = rows
    if r2 is None:
        top_set = bot_set = allb = r1
    else:
        top_set, bot_set = (r1, r2) if np.median(r1[:, 1]) < np.median(r2[:, 1]) else (r2, r1)
        allb = np.vstack([r1, r2])
    hs = np.median(allb[:, 3])
    kt = _fit(top_set[:, 0] + top_set[:, 2] / 2, top_set[:, 1])                 # đường qua đỉnh ký tự
    kb = _fit(bot_set[:, 0] + bot_set[:, 2] / 2, bot_set[:, 1] + bot_set[:, 3])  # đường qua chân ký tự
    span = (allb[:, 0] + allb[:, 2]).max() - allb[:, 0].min()
    pad_x = max(1.0 * np.median(allb[:, 2]), 0.08 * span)              # lề ngang: ký tự phía xa thường bị sót
    xl = max(0.0, allb[:, 0].min() - pad_x)
    xr = min(gray.shape[1] - 1.0, (allb[:, 0] + allb[:, 2]).max() + pad_x)
    mg = 0.25 * hs
    q = np.float32([[xl, np.polyval(kt, xl) - mg], [xr, np.polyval(kt, xr) - mg],
                    [xr, np.polyval(kb, xr) + mg], [xl, np.polyval(kb, xl) + mg]]) / scale
    q[:, 0] = np.clip(q[:, 0], 0, img_rgb.shape[1] - 1)
    q[:, 1] = np.clip(q[:, 1], 0, img_rgb.shape[0] - 1)
    h_px = max(np.linalg.norm(q[3] - q[0]), np.linalg.norm(q[2] - q[1]))
    w_px = max(np.linalg.norm(q[1] - q[0]), np.linalg.norm(q[2] - q[3]))
    if h_px < 5 or w_px < 10:
        return None
    out_w = int(round(out_h * w_px / h_px))
    M = cv2.getPerspectiveTransform(q, np.float32([[0, 0], [out_w, 0], [out_w, out_h], [0, out_h]]))
    return cv2.warpPerspective(img_rgb, M, (out_w, out_h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


# ---------- Chỉnh sáng/tối: tạo vài biến thể, giữ kết quả có độ tin cậy cao nhất ----------
def _gamma(img, g):
    return np.clip(255 * (img / 255.0) ** g, 0, 255).astype(np.uint8)

def _tone_variants(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l = lab[..., 0]
    clahe = lab.copy(); clahe[..., 0] = cv2.createCLAHE(2.5, (4, 4)).apply(l)
    return [img,
            cv2.cvtColor(clahe, cv2.COLOR_LAB2RGB),                    # tăng tương phản cục bộ
            _gamma(img, 0.5 if l.mean() < 110 else 1.8),               # ảnh tối -> sáng lên, ảnh chói -> tối xuống
            cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)]         # kéo giãn dải sáng

def _read_best(imgs):
    best = ('', 0.0)
    for im in imgs:
        inp = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if _PLATE_GRAY else im
        p = PLATE_OCR.run(inp, return_confidence=True)[0]
        cf = float(np.mean(p.char_probs)) if p.char_probs is not None and len(p.char_probs) else 0.0
        txt = re.sub(r'[^A-Z0-9]', '', p.plate.upper())
        if len(txt) >= 4 and cf > best[1]:                             # bỏ kết quả rỗng/quá ngắn
            best = (txt, cf)
    return best

def recognize_plate_text(img_rgb, box, reader=None, min_conf=0.5, rect_margin=0.02):
    """OCR v3.1: cắt biển (nới 5-8%) -> đọc bản gốc + bản LÀM PHẲNG, mỗi bản 4 mức sáng
    -> chỉ dùng bản làm phẳng khi tin cậy hơn bản gốc > rect_margin -> định dạng. Trả (chuỗi, conf)."""
    H, W = img_rgb.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    px, py = int(0.05 * (x2 - x1)), int(0.08 * (y2 - y1))
    crop = img_rgb[max(0, y1 - py):min(H, y2 + py), max(0, x1 - px):min(W, x2 + px)]
    if crop.size == 0:
        return '', 0.0
    text, conf = _read_best(_tone_variants(crop))
    flat = rectify_by_chars(crop)
    if flat is not None:
        t2, c2 = _read_best(_tone_variants(flat))
        if c2 > conf + rect_margin:
            text, conf = t2, c2
    if conf < min_conf:                         # biển quá nhỏ/mờ: bỏ, tránh in chuỗi sai
        return '', conf
    return _fmt_plate(text), conf

assert _fmt_plate('51F32488') == '51F-324.88' and _fmt_plate('59C312345') == '59C3-123.45'
assert _fmt_plate('BH5262') == 'BH5262'
print('Đã nạp OCR v3.1: recognize_plate_text(), rectify_by_chars() sẵn sàng cho các ô phía sau.')


ocr_reader = PLATE_OCR   # ô Gradio chỉ đọc biển số khi biến này tồn tại

Đã nạp OCR v3.1: recognize_plate_text(), rectify_by_chars() sẵn sàng cho các ô phía sau.


In [4]:
# CELL 4: Mở web demo Gradio (link public *.gradio.live)
import gradio as gr

print("\n--- Khoi tao Web Demo Gradio (YOLO + U-Net + ResNet) ---")
demo_yolo = YOLO(str(WEIGHTS_DIR / 'yolov8_best.pt'))

# U-Net (phan doan bien so) - TUY CHON: chi dung neu da huan luyen o Section 4.
demo_unet = None
_unet_w = WEIGHTS_DIR / 'unet_aug_best.pth'
try:
    if 'UNetSegmentation' in globals() and _unet_w.exists():
        demo_unet = UNetSegmentation().to(DEVICE)
        demo_unet.load_state_dict(torch.load(str(_unet_w), map_location=DEVICE, weights_only=True))
        demo_unet.eval()
        demo_unet_tf = transforms.Compose([
            transforms.Resize((256, 256)), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        print("  Da nap U-Net.")
    else:
        print("  U-Net chua co trong so -> dung lop phu bbox thay cho mask.")
except Exception as e:
    print("  Khong nap duoc U-Net:", e); demo_unet = None

# ResNet50 (phan loai loai phuong tien) - TUY CHON: chi dung neu da huan luyen o Section 2.
demo_cls = None
_cls_w = WEIGHTS_DIR / 'resnet50_vehicle_v2.pth'   # TN1 v2 (nhan YOLOv8-COCO, Acc 81.6%)
try:
    if 'VehicleResNet50' in globals() and _cls_w.exists():
        demo_cls = VehicleResNet50(NUM_CLASSES).to(DEVICE)
        demo_cls.load_state_dict(torch.load(str(_cls_w), map_location=DEVICE, weights_only=True))
        demo_cls.eval()
        demo_cls_tf = transforms.Compose([
            transforms.Resize((224, 224)), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        print("  Da nap ResNet50 phan loai phuong tien.")
    else:
        print("  ResNet chua co trong so -> bo qua phan loai phuong tien.")
except Exception as e:
    print("  Khong nap duoc ResNet:", e); demo_cls = None


def predict(input_image, conf_threshold=0.25):
    """Pipeline 3 model: YOLOv8 (phat hien) + U-Net (phan doan) + ResNet50 (phan loai)."""
    if input_image is None:
        return None, "{}"
    img_np = np.array(input_image)
    vis = img_np.copy()
    t_start = time.time()

    # 1) Phan loai loai phuong tien (toan anh) bang ResNet
    vehicle_type, vehicle_conf = None, None
    if demo_cls is not None:
        ct = demo_cls_tf(Image.fromarray(img_np)).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prob = torch.softmax(demo_cls(ct), dim=1)[0]
        k = int(prob.argmax()); vehicle_type = CLASS_NAMES[k]; vehicle_conf = float(prob[k])

    # 2) Phat hien bien so bang YOLO
    results = demo_yolo(img_np, conf=float(conf_threshold), verbose=False)[0]
    detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        _H, _W = img_np.shape[:2]
        x1 = max(0, x1); y1 = max(0, y1); x2 = min(_W, x2); y2 = min(_H, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        conf = float(box.conf[0])
        plate_text = ""
        if globals().get('ocr_reader') is not None:
            try:
                plate_text, _ = recognize_plate_text(img_np, (x1, y1, x2, y2))
            except Exception:
                plate_text = ""
        # 3) Phan doan bien so bang U-Net (neu co), khong thi to lop phu bbox
        used_unet = False
        if demo_unet is not None:
            crop = img_np[max(0, y1):y2, max(0, x1):x2]
            if crop.size > 0:
                ci = demo_unet_tf(Image.fromarray(crop)).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    m = demo_unet(ci).squeeze().cpu().numpy()
                mb = cv2.resize((m > 0.5).astype(np.uint8) * 255, (x2 - x1, y2 - y1))
                ov = np.zeros_like(vis[max(0, y1):y2, max(0, x1):x2]); ov[:, :, 1] = mb
                reg = vis[max(0, y1):y2, max(0, x1):x2]
                vis[max(0, y1):y2, max(0, x1):x2] = cv2.addWeighted(reg, 0.7, ov, 0.3, 0)
                used_unet = True
        if not used_unet:
            ov = vis.copy(); cv2.rectangle(ov, (x1, y1), (x2, y2), (0, 200, 0), -1)
            vis = cv2.addWeighted(ov, 0.25, vis, 0.75, 0)
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        _lbl = plate_text if plate_text else f'LP {conf:.2f}'
        cv2.putText(vis, _lbl, (x1, max(0, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        detections.append({'bbox': [x1, y1, x2, y2], 'confidence': round(conf, 4),
                           'label': 'license_plate', 'plate_text': plate_text})

    if vehicle_type is not None:
        cv2.putText(vis, f'{vehicle_type} {vehicle_conf:.2f}', (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 128, 0), 2)

    ms = (time.time() - t_start) * 1000
    out_json = json.dumps({
        'vehicle_type': vehicle_type,
        'vehicle_confidence': round(vehicle_conf, 4) if vehicle_conf is not None else None,
        'num_detections': len(detections),
        'inference_time_ms': round(ms, 1),
        'segmentation': ('U-Net' if demo_unet is not None else 'bbox'),
        'detections': detections,
    }, indent=2, ensure_ascii=False)
    return vis, out_json


_ex = None
try:
    _first = next(Path(TEST_IMG_DIR).glob('*.jpg'), None)
    if _first is not None:
        _ex = [[str(_first), 0.25]]
except Exception:
    _ex = None

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="pil", label="Upload anh phuong tien giao thong"),
        gr.Slider(0.05, 0.95, value=0.35, step=0.05, label="Confidence Threshold"),
    ],
    outputs=[
        gr.Image(type="numpy", label="Ket qua: Detection + Segmentation + Classification"),
        gr.Textbox(label="JSON Output", lines=14),
    ],
    title="Nhan dien Phuong tien & Bien so xe - Nhom 1 (YOLOv8 + U-Net + ResNet50)",
    description=(
        "Nhom 1 - Tri tue nhan tao nang cao & ung dung - DH Ba Ria - Vung Tau. "
        "YOLOv8n phat hien bien so, U-Net phan doan, ResNet50 phan loai phuong tien. "
        "Khong can API key, khong can localhost."
    ),
    examples=_ex,
)
demo.launch(inline=True, share=True, debug=False)
print("  Gradio demo dang chay inline + co link cong khai!")
print("\n📌 CHÈN VÀO BÁO CÁO → Chương 5 (giao diện demo & kết quả suy luận 3 model).")


--- Khoi tao Web Demo Gradio (YOLO + U-Net + ResNet) ---
  Da nap U-Net.
  Da nap ResNet50 phan loai phuong tien.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bcf9ac578b569c895f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  Gradio demo dang chay inline + co link cong khai!

📌 CHÈN VÀO BÁO CÁO → Chương 5 (giao diện demo & kết quả suy luận 3 model).
